|<img style="float:left;" src="../images/usherb_transp.gif" width=90% height=90%> |<big>Pierre Proulx, ing, professeur</big>|
|:---|:---|
|Département de génie chimique et de génie biotechnologique |** GCH200-Phénomènes d'échanges I **|



## Exercice 7A6

<img src='../images/GCH200-Chap7-A6.png' width = 50% height=50%>

#### Solution

Appliquons entre les points 1 et 2 étant définis comme la surface du réservoir du haut et la sortie de la conduite du bas.

La longueur totale des tuyaux est de 12' + 11' + 14 ', soit 37 pieds soit 11.28 mètres. La différence de hauteur entre les deux points est de 38 pieds, soit 11.58 m. Le diamètre des tuyaux est de 5'' soit 12.7 cm. La densité de l'eau est 998.2 kg/m3 à 20 degrés et sa viscosité de 0.001002 Pa-s. 

Les deux pressions aux points 1 et 2 sont la pression atmosphérique, posée à 0.

L'équation de bilan d'énergie mécanique est:

<center>
$\begin{equation}
  \boxed{ \sum_{entrée} \bigg ( \frac {1}{2} v_1^2 + g h_1 + \frac {p_1}{\rho_1} \bigg)w_1-\sum_{sortie} \bigg ( \frac {1}{2} v_2^2 + g h_2 + \frac {p_2}{\rho_2} \bigg )w_2=-W_m +E_c+ \sum_{i-conduites} (w_i 2 \frac  {L}{D} v^2 f)_i + \sum_{j-perturbations} (w_i \frac  {1}{2} v^2 e_v)_j}
  \end{equation}$
</center>

En l'appliquant au cas présent, on peut diviser par le débit car il est identique partout::

<center>
$\begin{equation}
  \boxed{  \big ( 0  + gh_1 + 0  \bigg)-\big ( \frac {1}{2} v_2^2 + 0 + 0 \bigg )=2 \big (( \frac  {L}  {D}    v_2   ^2 f)\big )+\big ( \frac {1}{2} v_2^2 ( 0.45 + 0.4 + 0.4 ) \big )
         }
  \end{equation}$
</center>

On fera l'hypothèse que l'écoulement est turbulent et que la conduite est lisse, alors f est donné par l'équation de Blasius (valide pour Re< 100000 environ):

<center>
$\begin{equation*}
  \boxed{  \big ( 0 + 0 + g \times h_2 \bigg)-\big ( \frac {1}{2} v_2^2 \bigg ) =2 \big (( \frac  {L}  {D}    v_2   ^2 \frac {0.0791 \mu} {\rho v_2 D})\big )+\big ( \frac {1}{2} v_2^2 ( 0.45 + 0.4 + 0.4 )\big )
         }
  \end{equation*}$
</center>

La seule inconnue est $v_2$, on peut solutionner facilement:

In [2]:
import thermo as th
from math import *
from fluids.units import *
from thermo.units import Stream ## attention, il faut les versions récentes de fluids et thermo (0.1.68 et 0.1.39)
#
# Exercice 7A6 BSL
#
Q=10*u.feet**3/u.min
Re=1
# On fait un estimé de départ pour le Reynolds et le facteur de friction puisqu'on sait pas à priori
Ren=100000                          # valeur estimée, pas d'effet sur le reste mais pas 0!!!
T = 68*u.degF
P = 1*u.bar
mu = 1*u.cP
r_d=0.65
rho= 62.4*u.lb/u.feet**3
water = Stream('eau', T=T, P=P, Q=Q)
NPS, Di, Do, t = nearest_pipe(Di=5*u.inch)
v = Q/(pi*Di**2/4)
fd=friction_factor(Re=Ren)
L=(12+11+14)*u.feet
H=(12+14+12)*u.feet
# On va ensuite faire des itérations pour converger vers la solution, car l'estimé de départ est assurément faux
compteur=0
precision=1e-2
while (abs((Re-Ren)/Re))>precision:
    compteur+=1
    Re=Ren
    # terme associé à la friction sur la conduite de longeur L
    K = K_from_f(fd=fd, L=L, D=Di)
    # termes de ev, friction associée aux coudes, entrée, sortie, ou autres
    K +=2*bend_rounded(Di=Di, angle=90*u.degrees, fd=fd, bend_diameters=r_d)
    K += entrance_sharp()
    K += exit_normal()
    v=(2*u.gravity*H/K)**0.5 # u.gravity est l'accélération gravitationelle en m/s^2 dans le système SI, mais pas en système anglais...
    Q=v*pi/4*Di**2
    Ren = Reynolds(D=Di, rho=water.rho, mu=water.mu, V=v)
    fd=friction_factor(Re=Ren)
    print(fd,Ren)
print ('Convergence à',precision, 'après',compteur,' itérations')
print('Re=',Ren,' Débits=',Q.to(u.gallon/u.min),Q.to(u.liter/u.min),Q.to(u.cups/u.min),Q.to(u.ft**3/u.hr),Q.to(u.barrel/u.day))
print(v)

0.011785593331312287 dimensionless 932574.4099646246 dimensionless
0.011554207148374972 dimensionless 1046833.4603237964 dimensionless
0.011544597165762099 dimensionless 1051947.1309550754 dimensionless
Convergence à 0.01 après 3  itérations
Re= 1051947.1309550754 dimensionless  Débits= 1684.970246495682 gallon / minute 6378.306226774139 liter / minute 26959.52394393091 cup / minute 13514.865518767452 foot ** 3 / hour 77027.21126837404 barrel / day
4.763436333309846 foot ** 0.5 * standard_gravity ** 0.5
